# 07 — SARIMAX com variáveis externas

Nos notebooks anteriores, treinamos ARIMA e SARIMA usando apenas a própria série de demanda.

Agora vamos testar uma evolução: o SARIMAX.

O SARIMAX permite adicionar variáveis externas ao modelo, como:

- temperatura;
- umidade;
- vento;
- feriado;
- dia útil;
- dia da semana.

A pergunta agora é:

> será que o modelo melhora quando usamos informações do contexto?

### 07.01 Imports e caminhos

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

import plotly.graph_objects as go

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

if Path.cwd().name == "notebooks":
    raiz_projeto = Path.cwd().parent
else:
    raiz_projeto = Path.cwd()

sys.path.append(str(raiz_projeto / "src"))

from visual_utils import (
    CORES,
    aplicar_layout_padrao,
    grafico_linha_padrao
)

caminho_outputs_tabelas = raiz_projeto / "outputs" / "tabelas"

caminho_base_organizada = caminho_outputs_tabelas / "base_bike_organizada.csv"
caminho_comparacao_modelos = caminho_outputs_tabelas / "comparacao_modelos.csv"
caminho_previsoes_arima = caminho_outputs_tabelas / "previsoes_arima.csv"
caminho_previsoes_sarima = caminho_outputs_tabelas / "previsoes_sarima.csv"

print("Base organizada:")
print(caminho_base_organizada)

### 07.02 Carregar base organizada

In [ ]:
df = pd.read_csv(caminho_base_organizada)

df["data_hora"] = pd.to_datetime(df["data_hora"])

df = df.sort_values("data_hora").reset_index(drop=True)

print(f"Linhas: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")

df.head()

## Por que usar variáveis externas?

Até agora, ARIMA e SARIMA usaram apenas o histórico da demanda.

Mas, no mundo real, a demanda de bicicletas pode depender também de fatores externos.

Por exemplo:

- em dias muito frios, a demanda pode cair;
- em dias úteis, o padrão pode ser diferente;
- em feriados, o comportamento pode mudar;
- clima e umidade também podem influenciar o uso.

### 07.03 Criar base diária com variáveis externas

In [ ]:
base_diaria = (
    df.assign(data=df["data_hora"].dt.floor("D"))
    .groupby("data", as_index=False)
    .agg(
        demanda=("demanda", "sum"),
        temperatura_media=("temperatura", "mean"),
        sensacao_termica_media=("sensacao_termica", "mean"),
        umidade_media=("umidade", "mean"),
        velocidade_vento_media=("velocidade_vento", "mean"),
        feriado=("feriado", "max"),
        dia_util=("dia_util", "max"),
        dia_semana=("dia_semana", "first"),
        mes=("mes", "first")
    )
    .rename(columns={"data": "data_hora"})
)

print(f"Quantidade de dias observados: {base_diaria.shape[0]}")

base_diaria.head()

### 07.04 Criar variáveis de dia da semana

In [ ]:
dummies_dia_semana = pd.get_dummies(
    base_diaria["dia_semana"],
    prefix="dia_semana",
    drop_first=True
)

base_modelagem = pd.concat(
    [
        base_diaria,
        dummies_dia_semana
    ],
    axis=1
)

base_modelagem.head()

### 07.05 Definir variáveis externas

In [ ]:
variaveis_externas = [
    "temperatura_media",
    "sensacao_termica_media",
    "umidade_media",
    "velocidade_vento_media",
    "feriado",
    "dia_util"
] + list(dummies_dia_semana.columns)

print("Variáveis externas usadas no SARIMAX:")
variaveis_externas

### 07.06 Separar treino e teste

In [ ]:
tamanho_teste = 60

treino = base_modelagem.iloc[:-tamanho_teste].copy()
teste = base_modelagem.iloc[-tamanho_teste:].copy()

y_treino = treino["demanda"].astype(float)
y_teste = teste["demanda"].astype(float)

X_treino = treino[variaveis_externas].astype(float)
X_teste = teste[variaveis_externas].astype(float)

print("Tamanho do treino:", treino.shape[0])
print("Tamanho do teste:", teste.shape[0])

print("\nPeríodo de treino:")
print(treino["data_hora"].min(), "até", treino["data_hora"].max())

print("\nPeríodo de teste:")
print(teste["data_hora"].min(), "até", teste["data_hora"].max())

## Treinando o SARIMAX

O SARIMAX mantém a lógica do SARIMA, mas adiciona variáveis externas.

Vamos começar com a mesma estrutura sazonal do notebook anterior:

- `order = (1, 1, 1)`
- `seasonal_order = (1, 0, 1, 7)`

A diferença agora é que o modelo também recebe `X_treino`.

### 07.07 Treinar SARIMAX

In [ ]:
order = (1, 1, 1)
seasonal_order = (1, 0, 1, 7)

modelo_sarimax = SARIMAX(
    y_treino,
    exog=X_treino,
    order=order,
    seasonal_order=seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)

resultado_sarimax = modelo_sarimax.fit(disp=False)

resultado_sarimax.summary()

### 07.08 Gerar previsões no teste

In [ ]:
previsao_sarimax = resultado_sarimax.forecast(
    steps=len(teste),
    exog=X_teste
)

teste_resultado = teste[["data_hora", "demanda"]].copy()
teste_resultado["previsao_sarimax"] = previsao_sarimax.values

teste_resultado.head()

### 07.09 Visualizar real versus SARIMAX

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=teste_resultado["data_hora"],
        y=teste_resultado["demanda"],
        mode="lines",
        name="Real",
        line=dict(color=CORES["azul_escuro"], width=3)
    )
)

fig.add_trace(
    go.Scatter(
        x=teste_resultado["data_hora"],
        y=teste_resultado["previsao_sarimax"],
        mode="lines",
        name="SARIMAX",
        line=dict(color=CORES["azul_principal"], width=3)
    )
)

fig = aplicar_layout_padrao(
    fig,
    titulo="Real versus previsto — SARIMAX",
    altura=500
)

fig.show()

### 07.10 Função de métricas

In [ ]:
def calcular_metricas(y_real, y_pred):
    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))

    y_real = np.array(y_real)
    y_pred = np.array(y_pred)

    mascara = y_real != 0
    mape = np.mean(
        np.abs((y_real[mascara] - y_pred[mascara]) / y_real[mascara])
    ) * 100

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape
    }

### 07.11 Calcular métricas do SARIMAX

In [ ]:
metricas_sarimax = calcular_metricas(
    teste_resultado["demanda"],
    teste_resultado["previsao_sarimax"]
)

df_metricas_sarimax = pd.DataFrame([
    {
        "modelo": "SARIMAX(1,1,1)(1,0,1,7)",
        **metricas_sarimax
    }
])

df_metricas_sarimax

### 07.12 Carregar comparação anterior

In [ ]:
comparacao_modelos = pd.read_csv(caminho_comparacao_modelos)

comparacao_modelos

### 07.13 Comparar todos os modelos

In [ ]:
comparacao_final = pd.concat(
    [
        comparacao_modelos,
        df_metricas_sarimax
    ],
    ignore_index=True
)

comparacao_final = (
    comparacao_final
    .sort_values("MAPE")
    .reset_index(drop=True)
)

comparacao_final

### 07.14 Visualizar comparação final

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=comparacao_final["modelo"],
        y=comparacao_final["MAPE"],
        marker_color=CORES["azul_principal"],
        name="MAPE"
    )
)

fig = aplicar_layout_padrao(
    fig,
    titulo="Comparação final dos modelos pelo MAPE",
    altura=450,
    legenda_horizontal=False
)

fig.update_xaxes(title="Modelo")
fig.update_yaxes(title="MAPE (%)")

fig.show()

#### 07.15 Melhor modelo

In [ ]:
melhor_modelo = comparacao_final.iloc[0]

nome_melhor_modelo = melhor_modelo["modelo"]

print("Melhor modelo na comparação final:")
print(nome_melhor_modelo)

print("\nMAPE:")
print(f"{melhor_modelo['MAPE']:.2f}%")

## Atenção sobre previsão futura

O SARIMAX usa variáveis externas.

Isso melhora o modelo, mas traz uma responsabilidade:

> para prever o futuro, precisamos ter também valores futuros das variáveis externas.

Por exemplo, se usamos temperatura e umidade, precisamos ter uma previsão ou um cenário para essas variáveis.

Neste projeto, vamos salvar a comparação final e as previsões do teste. A previsão futura poderá ser apresentada como um cenário simples na etapa final.

### 07.16 Consolidar previsões do teste

In [ ]:
previsoes_arima = pd.read_csv(caminho_previsoes_arima)
previsoes_sarima = pd.read_csv(caminho_previsoes_sarima)

previsoes_arima["data_hora"] = pd.to_datetime(previsoes_arima["data_hora"])
previsoes_sarima["data_hora"] = pd.to_datetime(previsoes_sarima["data_hora"])

previsoes_teste = previsoes_arima[
    [
        "data_hora",
        "demanda",
        "previsao_baseline",
        "previsao_arima"
    ]
].copy()

previsoes_teste = previsoes_teste.merge(
    previsoes_sarima[["data_hora", "previsao_sarima"]],
    on="data_hora",
    how="left"
)

previsoes_teste = previsoes_teste.merge(
    teste_resultado[["data_hora", "previsao_sarimax"]],
    on="data_hora",
    how="left"
)

previsoes_teste = previsoes_teste.rename(
    columns={"demanda": "demanda_real"}
)

previsoes_teste.head()

### 07.17 Visualizar previsões no teste

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=previsoes_teste["data_hora"],
        y=previsoes_teste["demanda_real"],
        mode="lines",
        name="Real",
        line=dict(color="#0B1F4D", width=3)
    )
)

fig.add_trace(
    go.Scatter(
        x=previsoes_teste["data_hora"],
        y=previsoes_teste["previsao_baseline"],
        mode="lines",
        name="Baseline",
        line=dict(color="#7A7A7A", width=2, dash="dash")
    )
)

fig.add_trace(
    go.Scatter(
        x=previsoes_teste["data_hora"],
        y=previsoes_teste["previsao_arima"],
        mode="lines",
        name="ARIMA",
        line=dict(color="#6FA8FF", width=2)
    )
)

fig.add_trace(
    go.Scatter(
        x=previsoes_teste["data_hora"],
        y=previsoes_teste["previsao_sarima"],
        mode="lines",
        name="SARIMA",
        line=dict(color="#F28E2B", width=2)
    )
)

fig.add_trace(
    go.Scatter(
        x=previsoes_teste["data_hora"],
        y=previsoes_teste["previsao_sarimax"],
        mode="lines",
        name="SARIMAX",
        line=dict(color="#2CA02C", width=3)
    )
)

fig = aplicar_layout_padrao(
    fig,
    titulo="Comparação das previsões no período de teste",
    altura=500
)

fig.show()

### 07.18 Comparação Final

In [ ]:
baseline_mape = comparacao_final.loc[
    comparacao_final["modelo"] == "Baseline",
    "MAPE"
].iloc[0]

sarimax_mape = comparacao_final.loc[
    comparacao_final["modelo"].str.contains("SARIMAX"),
    "MAPE"
].iloc[0]

reducao_mape = ((baseline_mape - sarimax_mape) / baseline_mape) * 100

print(f"MAPE da baseline: {baseline_mape:.2f}%")
print(f"MAPE do SARIMAX: {sarimax_mape:.2f}%")
print(f"Redução aproximada do erro percentual: {reducao_mape:.1f}%")

### 07.18 Salvar resultados finais

In [ ]:
caminho_metricas_final = caminho_outputs_tabelas / "metricas.csv"
caminho_previsoes_teste_final = caminho_outputs_tabelas / "previsoes_teste.csv"
caminho_metricas_sarimax = caminho_outputs_tabelas / "metricas_sarimax.csv"
caminho_previsoes_sarimax = caminho_outputs_tabelas / "previsoes_sarimax.csv"
caminho_serie_historica = caminho_outputs_tabelas / "serie_historica.csv"

comparacao_final.to_csv(
    caminho_metricas_final,
    index=False,
    encoding="utf-8-sig"
)

df_metricas_sarimax.to_csv(
    caminho_metricas_sarimax,
    index=False,
    encoding="utf-8-sig"
)

teste_resultado.to_csv(
    caminho_previsoes_sarimax,
    index=False,
    encoding="utf-8-sig"
)

previsoes_teste.to_csv(
    caminho_previsoes_teste_final,
    index=False,
    encoding="utf-8-sig"
)

base_diaria[["data_hora", "demanda"]].to_csv(
    caminho_serie_historica,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos salvos:")
print("-", caminho_metricas_final)
print("-", caminho_previsoes_teste_final)
print("-", caminho_metricas_sarimax)
print("-", caminho_previsoes_sarimax)
print("-", caminho_serie_historica)

## Leitura do resultado

O SARIMAX apresentou o menor erro entre os modelos testados.

Isso mostra que, para este problema, variáveis externas como temperatura, umidade, vento, feriado e dia útil ajudam o modelo a representar melhor a demanda.

A principal lição é: modelos de séries temporais não dependem apenas do algoritmo, mas também da qualidade das informações usadas.

## Próximo passo

Agora temos uma comparação mais completa.

Vimos que ARIMA e SARIMA usam apenas o passado da série, enquanto o SARIMAX permite adicionar informações externas do problema.

Na próxima etapa, vamos organizar os arquivos finais para apresentar o projeto em uma aplicação Streamlit.